# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [5]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [6]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [7]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
openai_key = os.getenv("CHROMA_OPENAI_API_KEY")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    api_base=os.getenv("OPENAI_BASE_URL")
)

In [10]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [11]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [13]:
results = collection.query(
    query_texts=["minecraft"],
    n_results=3
)

for i in range(len(results["ids"][0])):
    print(f"ID: {results['ids'][0][i]}")
    print(f"Document: {results['documents'][0][i]}")
    print(f"Metadata: {results['metadatas'][0][i]}")
    print(f"Distance: {results['distances'][0][i]}")
    print("---")

ID: 014
Document: [Xbox One] Minecraft (2014) - A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.
Metadata: {'Genre': 'Sandbox, Survival', 'Name': 'Minecraft', 'YearOfRelease': 2014, 'Publisher': 'Mojang Studios', 'Description': 'A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.', 'Platform': 'Xbox One'}
Distance: 0.3153514266014099
---
ID: 009
Document: [Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
Metadata: {'Genre': 'Platformer', 'Publisher': 'Nintendo', 'Name': 'Super Mario 64', 'Platform': 'Nintendo 64', 'YearOfRelease': 1996, 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."}
Distance: 0.42526090145111084
---
ID: 008
Document: [Super Nintendo Entertainment System (SNES)]